<h1><b>Modul Pembelajaran Informed Search RKA 25</b></h1>

In [ ]:
from data.utils import idastar_search, rbfs_search, sma_star_search
import inspect

In [2]:
def psource(*functions):
    "Print the source code for the given function(s)."
    code = '\n\n'.join(inspect.getsource(fn) for fn in functions)
    print(code)

<h2><b>1. Iterative Deepening A* (IDA*)</b></h2>
Iterative Deepening A* (IDA*) menggabungkan efisiensi penelusuran ruang memori dari Depth-First Search dengan tingkat optimalitas algoritma A*. Algoritma ini berjalan dalam sebuah loop berulang di mana setiap iterasinya menelusuri lintasan graf layaknya DFS, tetapi ia menetapkan batas biaya (<i>cost threshold</i>) berdasarkan f(n) = g(n) + h(n). 

Mekanisme setiap langkah IDA*:
- Set threshold awal = nilai $f(n)$ dari root node
- Jalankan DFS, potong (cutoff) jalur jika nilai $f(n) >$ threshold
- Jika goal ditemukan → selesai
- Jika tidak → set threshold baru = nilai $f(n)$ minimum yang melampaui threshold sebelumnya
- Ulangi proses dengan threshold baru

Kelebihan IDA*:
- Efisien memori = $O(bd)$, meniru memori linier DFS
- Complete dan Optimal = Pasti menemukan cost terendah (jika heuristik admissible)

Kekurangan IDA*:
- Re-computation = Node yang sama sering dikunjungi berkali-kali pada setiap iterasi
- Overhead iterasi = Kurang efisien jika setiap node memiliki nilai heuristik yang unik


![](data\ida_star.gif)

Sumber GIF : https://algorithmsinsight.wordpress.com/wp-content/uploads/2016/03/ida-star.gif

<b>Implementasi IDA*</b>

Struktur Graf dengan heuristik:
- Graf memuat nilai pergerakan asli (g).
- Heuristik memuat batas bawah perkiraan sisa menuju <i>goal</i> (h).

In [6]:
psource(idastar_search)

def idastar_search(graph, heuristics, start, goal):
    threshold = heuristics[start]
    
    def search(node, g, threshold, path):
        f = g + heuristics[node]
        if f > threshold:
            return f, None
        if node == goal:
            return -1, path + [node]
            
        min_threshold = float('inf')
        for neighbor, cost in graph.get(node, []):
            if neighbor not in path:
                temp_curr, result_path = search(neighbor, g + cost, threshold, path + [node])
                if temp_curr == -1:
                    return -1, result_path
                if temp_curr < min_threshold:
                    min_threshold = temp_curr
        return min_threshold, None

    path = []
    while True:
        temp, result_path = search(start, 0, threshold, path)
        if temp == -1:
            return result_path, threshold
        if temp == float('inf'):
            return None, float('inf')
        threshold = temp



In [2]:
graph_ida = {
    'S': [('A', 2), ('B', 5)],
    'A': [('C', 2), ('D', 4)],
    'B': [('D', 1)],
    'C': [('G', 2)],
    'D': [('G', 3)],
    'G': []
}

heuristics_ida = {
    'S': 6, 'A': 4, 'B': 4, 'C': 2, 'D': 2, 'G': 0
}

print("IDA* Path:", idastar_search(graph_ida, heuristics_ida, 'S', 'G')[0])

IDA* Path: ['S', 'A', 'C', 'G']


<b>PR IDA* Algorithm</b>

Fino berada di koordinat (0, 0) dan harus menuju lokasi Naga di koordinat (4, 4) pada sebuah matriks grid 5x5. Grid ini memiliki rintangan bernilai 1 yang tidak dapat dilalui, sedangkan area bebas bernilai 0. Temukan jalur terpendek menggunakan algoritma IDA* dengan heuristik jarak Manhattan.

In [ ]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def get_neighbors(grid, r, c):
    neighbors = []
    for dr, dc in [(-1,0), (1,0), (0,-1), (0,1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]) and grid[nr][nc] == 0:
            neighbors.append((nr, nc))
    return neighbors

def idastar_grid(grid, start, goal):
    def search(node, g, threshold, path):
        f = g + manhattan(node, goal)
        if f > ___:
            return f, None
        if node == ___:
            return -1, path + [node]
            
        min_threshold = float('inf')
        for neighbor in get_neighbors(grid, node[0], node[1]):
            if neighbor not in path:
                temp, result_path = search(___, g + 1, ___, path + [node])
                if temp == -1: 
                    return -1, result_path
                if temp < min_threshold: 
                    min_threshold = ___
        return min_threshold, None

    threshold = manhattan(start, goal)
    while True:
        temp, result_path = search(start, 0, threshold, [])
        if temp == -1: 
            return result_path
        if temp == float('inf'): 
            return None
        threshold = ___

In [ ]:
grid_1 = [
    [0, 0, 1, 0, 0],
    [0, 1, 0, 0, 1],
    [0, 0, 0, 1, 0],
    [1, 1, 0, 0, 0],
    [0, 0, 0, 1, 0]
]
path_ida = idastar_grid(grid_1, (0,0), (4,4))

<h2><b>2. Recursive Best-First Search (RBFS)</b></h2>
Recursive Best-first Search (RBFS) meniru perilaku standar pencarian Best-First Search namun menggunakan kapasitas memori linear yang menelusuri simpul paling menjanjikan secara rekursif, dilengkapi mekanisme backtracking untuk mengevaluasi jalur alternatif.

Mekanisme setiap langkah RBFS:
- Eksplorasi node mengikuti jalur dengan nilai $f(n)$ terkecil
- Simpan nilai $f(n)$ alternatif terbaik (node saudara) di setiap langkah
- Jika nilai $f(n)$ node saat ini melampaui nilai alternatif → lakukan backtrack
- Saat backtrack, perbarui nilai $f(n)$ node induk dengan nilai terbaik dari anak-anaknya
- Pindah eksplorasi ke jalur alternatif tersebut

Kelebihan RBFS:
- Efisien memori = Menggunakan ruang $O(bd)$
- Optimal = Selama heuristik yang digunakan admissible

Kekurangan RBFS:
- Thrashing (re-computation ekstrem) = Terlalu sering bolak-balik jalur pencarian
- Tidak memanfaatkan sisa memori = Tetap menggunakan sedikit memori meskipun tersedia besar

<b>Implementasi RBFS</b>

In [3]:
psource(rbfs_search)

def rbfs_search(graph, heuristics, start, goal):
    def rbfs(node, node_f, f_limit, path):
        if node == goal:
            return True, path + [node], node_f
            
        successors = graph.get(node, [])
        if not successors:
            return False, [], float('inf')
            
        succ_nodes = []
        for neighbor, cost in successors:
            total_g = len(path) * 1 
            child_f = max(total_g + cost + heuristics[neighbor], node_f)
            succ_nodes.append([child_f, neighbor, cost])
            
        while True:
            succ_nodes.sort(key=lambda x: x[0])
            best_f, best_node, best_cost = succ_nodes[0]
            
            if best_f > f_limit:
                return False, [], best_f
                
            alt_f = succ_nodes[1][0] if len(succ_nodes) > 1 else float('inf')
            
            result_bool, result_path, new_best_f = rbfs(
                best_node, best_f, min(f_limit, alt_f), path + [node]
      

In [4]:
graph_rbfs = {
    'N1': [('N2', 3), ('N3', 2)],
    'N2': [('N4', 4)],
    'N3': [('N4', 1), ('N5', 6)],
    'N4': [('N5', 2)],
    'N5': []
}

heuristics_rbfs = {
    'N1': 5, 'N2': 4, 'N3': 3, 'N4': 2, 'N5': 0
}

print("RBFS Path:", rbfs_search(graph_rbfs, heuristics_rbfs, 'N1', 'N5'))

RBFS Path: ['N1', 'N3', 'N4', 'N5']


**PR RBFS Algorithm**

Setelah bertemu di titik (4, 4), Naga dan Fino harus melanjutkan perjalanan menuju titik evakuasi di (0, 5) pada matriks perluasan 6x6. Area ini memiliki rintangan dan jarak antar simpul bernilai seragam. Gunakan algoritma RBFS untuk mengevaluasi batas nilai f(n) dan menemukan rute dengan biaya minimum secara rekursif.

In [ ]:
def rbfs_grid(grid, start, goal):
    def rbfs(node, node_f, f_limit, path, g):
        if node == goal: 
            return True, path + [node], node_f
        
        neighbors = get_neighbors(grid, node[0], node[1])
        if not neighbors: 
            return False, [], float('inf')
        
        succ = []
        for nbr in neighbors:
            if nbr not in path:
                child_f = max(g + 1 + manhattan(___, ___), ___)
                succ.append([child_f, nbr, 1])
                
        if not succ: 
            return False, [], float('inf')
        
        while True:
            succ.sort(key=lambda x: x[___])
            best_f, best_node, best_cost = succ[0]
            
            if best_f > ___: 
                return False, [], best_f
                
            alt_f = succ[1][0] if len(succ) > 1 else float('inf')
            res, res_path, best_f_new = rbfs(best_node, best_f, min(___, ___), path + [node], g + best_cost)
            
            if res: 
                return True, res_path, best_f_new
            succ[0][0] = ___

    res, path, _ = rbfs(start, manhattan(start, goal), float('inf'), [], 0)
    return path if res else None

In [ ]:
grid_2 = [
    [0, 0, 0, 1, 0, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 1, 1, 1, 0, 1],
    [0, 0, 0, 1, 0, 0],
    [1, 1, 0, 0, 0, 0],
    [0, 0, 0, 1, 1, 0]
]
path_rbfs = rbfs_grid(grid_2, (4,4), (0,5))

<h2><b>3. Simplified Memory Bounded A* (SMA*)</b></h2>
Variasi algoritma A* yang memanfaatkan seluruh ruang memori tetap dengan menghapus node terburuk ketika kapasitas memori penuh.

Mekanisme setiap langkah SMA*:
- Jalankan A* biasa dengan mengekspansi node berbiaya $f(n)$ terkecil
- Jika batas memori tercapai → hapus (prune) leaf node dengan nilai $f(n)$ tertinggi
- Simpan (backup) nilai node yang dihapus ke node induk
- Jika diperlukan, node dapat diregenerasi kembali
- Ulangi proses

Kelebihan SMA*:
- Penggunaan memori optimal = Memanfaatkan memori maksimum yang tersedia
- Complete = Jika memori cukup untuk menyimpan jalur solusi
- Optimal = Jika memori cukup untuk menyimpan solusi terbaik

Kekurangan SMA*:
- Overhead waktu sangat tinggi = Banyak proses hapus dan regenerasi node
- Rentan thrashing = Jika batas memori terlalu kecil dibanding kompleksitas masalah

<b>Implementasi SMA*</b>

In [5]:
psource(sma_star_search)

def sma_star_search(graph, heuristics, start, goal, max_memory=5):
    open_list = [(heuristics[start], 0, [start])]
    
    while open_list:
        open_list.sort(key=lambda x: (x[0], -len(x[2])))
        current_f, current_g, path = open_list.pop(0)
        current_node = path[-1]
        
        if current_node == goal:
            return path, current_f
            
        successors = graph.get(current_node, [])
        for neighbor, cost in successors:
            if neighbor not in path:
                g_new = current_g + cost
                f_new = max(current_f, g_new + heuristics[neighbor])
                open_list.append((f_new, g_new, path + [neighbor]))
                
        if len(open_list) > max_memory:
            open_list.sort(key=lambda x: (x[0], -len(x[2])))
            open_list.pop() 
            
    return None, float('inf')



In [6]:
graph_sma = {
    'R1': [('R2', 1), ('R3', 2)],
    'R2': [('R4', 3)],
    'R3': [('R4', 1), ('R5', 2)],
    'R4': [('R6', 2)],
    'R5': [('R6', 3)],
    'R6': []
}

heuristics_sma = {
    'R1': 4, 'R2': 3, 'R3': 3, 'R4': 2, 'R5': 2, 'R6': 0
}

print("SMA* Path:", sma_star_search(graph_sma, heuristics_sma, 'R1', 'R6', max_memory=4)[0])

SMA* Path: ['R1', 'R3', 'R4', 'R6']


<b>PR SMA* Algorithm</b>

Setelah evakuasi, Naga dan Fino memasuki labirin terakhir dengan ukuran 5x5 dari titik (0, 0) menuju (4, 4). Sistem navigasi mereka mengalami keterbatasan sehingga memori yang tersedia hanya mampu menampung batas maksimal 6 simpul secara bersamaan. Lakukan pencarian jalur yang mematuhi batasan memori tersebut menggunakan algoritma SMA*.

In [ ]:
def sma_star_grid(grid, start, goal, max_mem=6):
    open_list = [(manhattan(start, goal), 0, [start])]
    
    while open_list:
        open_list.sort(key=lambda x: (x[0], -len(x[___])))
        curr_f, curr_g, path = open_list.pop(0)
        curr_node = path[-1]
        
        if curr_node == ___: 
            return path
        
        for nbr in get_neighbors(grid, curr_node[0], curr_node[1]):
            if nbr not in path:
                g_new = curr_g + 1
                f_new = max(___, g_new + manhattan(___, goal))
                open_list.append((___, ___, path + [nbr]))
                
        if len(open_list) > ___:
            open_list.sort(key=lambda x: (x[0], -len(x[2])))
            open_list.___()
            
    return None

In [ ]:
grid_3 = [
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 1, 0],
    [1, 1, 0, 0, 0],
    [0, 0, 0, 1, 0]
]
path_sma = sma_star_grid(grid_3, (0,0), (4,4), max_mem=6)